In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import lm_eval
from lm_eval.tasks import TaskManager
from lm_eval.evaluator import simple_evaluate
from lm_eval.utils import make_table

# Constants

In [3]:
TAWJEEH_DATASET_NAME = 'xlsum'
HF_EXPERIMENTAL_DATASET_NAME = 'KFUPM-JRCAI/xlsum_arabic_experimental'
TASK_NAME='summarization'
MODEL_PATH = "/raid_storage/shared_models/Meta-Llama-3.1-8B"
TOKENIZER_PATH = MODEL_PATH
BATCH_SIZE = 64
# ------------------------
MODEL_NAME = MODEL_PATH.split('/')[-1]
# -----------------------
TUNED_MODEL_PATH = f"Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}"

# Building the prompts dataset

In [4]:
import requests
 
from tqdm.auto import tqdm
 
prompts = None
 
tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts:
    raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14888,
  'tags': ['AI generated'],
  'name': 'history_based_dialect',
  'task': {'name': 'dialect identification'},
  'status': 'SUBMITTED',
  'template': 'Consider this historical context in the text provided: {{Text}}. Which dialect from the ancient regions does it emerge from? Select from these regions: Levant, North Africa, Egypt, GULF, MSA. ||| {{answer_choices[label]}}',
  'dataset_name': 'arbml/Arabic_Dialects_Dataset',
  'dataset_subset': 'default',
  'answer_choices': ['Levant', 'North Africa', 'Egypt', 'GULF', 'MSA'],
  'text_direction': 'ltr'},
 {'id': 14887,
  'tags': ['AI generated'],
  'name': 'literary_style_identification',
  'task': {'name': 'dialect identification'},
  'status': 'SUBMITTED',
  'template': 'The narrative style found in the following phrase: {{Text}}, is tied to a specific dialect. Out of Levant, North Africa, Egypt, GULF, MSA, which one do you think it is? ||| {{answer_choices[label]}}',
  'dataset_name': 'arbml/Arabic_Dialects_Dataset',
  'dat

filter prompts:
- get only the approved ones
- get only the ones on the sarcasim detection datasets (emotone_ar,sem_eval_2018_task_1)

In [5]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

223

### Get the dataset prompts

In [6]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

7

In [7]:
SELECTED_PROMPTS_IDS = [
    14871,   
    14803,
    14856,
    14858,
    14668,
]

In [8]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

In [9]:
dataset_prompts

[{'id': 14871,
  'tags': [],
  'name': 'expert in summarization',
  'task': {'name': 'summarization'},
  'status': 'APPROVED',
  'template': 'You are an Arabic summarization expert! The summarization of the following Arabic passage: "{{text}}" is:\r\n|||\r\n{{target}}',
  'dataset_name': 'GEM/xlsum',
  'dataset_subset': 'arabic',
  'answer_choices': [],
  'text_direction': 'ltr'},
 {'id': 14858,
  'tags': [],
  'name': 'Simple summary generation',
  'task': {'name': 'summarization'},
  'status': 'APPROVED',
  'template': 'Generate a summary for the following Arabic document: {{text}}\r\n|||\r\n{{target}}',
  'dataset_name': 'GEM/xlsum',
  'dataset_subset': 'arabic',
  'answer_choices': [],
  'text_direction': 'ltr'},
 {'id': 14856,
  'tags': [],
  'name': 'Informative summry that captures the meaning',
  'task': {'name': 'summarization'},
  'status': 'APPROVED',
  'template': 'A concise and informative summary that captures the main points of the Arabic article "{{text}}" while maintai

### Download the experimental dataset

In [10]:
import datasets

In [11]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['gem_id', 'url', 'title', 'target', 'references', 'text'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['gem_id', 'url', 'title', 'target', 'references', 'text'],
        num_rows: 4689
    })
})

### Merge the prompts

In [12]:
from jinja2 import Environment, StrictUndefined

In [13]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        env = Environment(undefined=StrictUndefined)
        if "|||" not in template:
            raise ValueError("No ||| dividor")
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e
        

Perform generation on one example prompt, for experimentation

In [14]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['test'][1]))

You are an Arabic summarization expert! The summarization of the following Arabic passage: "قالت نافي إن تنظيم الدولة الإسلامية مارس انتهاكات جسيمة وفظيعة لحقوق الإنسان

وأوضحت بيلاي أن انتهاكات حقوق الإنسان شملت حالات قتل مستهدف وإرغام الناس على التحول إلى الإسلام والاختطافات والتهريب واستعباد الآخرين والانتهاكات الجنسية وتدمير أماكن ذات أهمية دينية وثقافية ومحاصرة مجتمعات محلية برمتها لأسباب إثنية أو دينية أو طائفية.

وأضافت بيلاي في بيان صادر عنها أن عدة مناطق في العراق تخضع لسيطرة الدولة الإسلامية وحلفائها شهدت "تطهيرا إثنيا ودينيا بدون هوادة".

وقالت مفوضة حقوق الإنسان إن "تنظيم الدولة الإسلامية والمجموعات المرتبطة به تمارس انتهاكات جسيمة وفظيعة لحقوق الإنسان بشكل يومي. إنهم يستهدفون بشكل منهجي رجالا ونساء وأطفالا بناء على إثنيتهم وولائهم الديني أو الطائفي".

ومضت بيلاي للقول إن مسيحيين وإيزيديين وتركمان من كانوا ضمن من استهدفهم تنظيم الدولة الإسلامية.

مواضيع قد تهمك نهاية

انتهاك منهجي

وقالت مفوضة حقوق الإنسان إن حالات الاضطهاد والانتهاك المنهجي لحقوق الإنسان التي وثقها محققو ا

In [15]:
for prompt in dataset_prompts:
    prompt['merged_samples'] = list(
        map(
            lambda sample: apply_template(prompt, sample),
            # hf_exp_dataset['test'].select(range(100)),
            tqdm(hf_exp_dataset['test']),
        )
    )
    prompt['original_samples'] = list(hf_exp_dataset['test'])

  0%|          | 0/4689 [00:00<?, ?it/s]

  0%|          | 0/4689 [00:00<?, ?it/s]

  0%|          | 0/4689 [00:00<?, ?it/s]

  0%|          | 0/4689 [00:00<?, ?it/s]

  0%|          | 0/4689 [00:00<?, ?it/s]

# Load LLM and Prepare

In [16]:
from datasets import DatasetDict
import re

def create_hf_dataset(dataset_prompt, columns=None):
  if columns is None:
    columns = ['text', 'summary']
  textes = []
  summaries = []
  for i,merged_sample in enumerate(dataset_prompt['merged_samples']):
    prefix= merged_sample.split('|||')[0]
    prefix = prefix.strip()
    prefix = prefix.replace('\xa0','')
    # prefix += '\nTranslation:'
    output = merged_sample.split('|||')[1].strip()
    textes.append(prefix)
    summaries.append(output)
  dataset = DatasetDict({ 'test' : datasets.Dataset.from_dict({
      columns[0]: textes,
      columns[1]: summaries,
  })})
  return dataset

In [17]:
dataset = create_hf_dataset(dataset_prompts[0])
dataset['test'][1],dataset['test'][1]

({'text': 'You are an Arabic summarization expert! The summarization of the following Arabic passage: "قالت نافي إن تنظيم الدولة الإسلامية مارس انتهاكات جسيمة وفظيعة لحقوق الإنسان\n\nوأوضحت بيلاي أن انتهاكات حقوق الإنسان شملت حالات قتل مستهدف وإرغام الناس على التحول إلى الإسلام والاختطافات والتهريب واستعباد الآخرين والانتهاكات الجنسية وتدمير أماكن ذات أهمية دينية وثقافية ومحاصرة مجتمعات محلية برمتها لأسباب إثنية أو دينية أو طائفية.\n\nوأضافت بيلاي في بيان صادر عنها أن عدة مناطق في العراق تخضع لسيطرة الدولة الإسلامية وحلفائها شهدت "تطهيرا إثنيا ودينيا بدون هوادة".\n\nوقالت مفوضة حقوق الإنسان إن "تنظيم الدولة الإسلامية والمجموعات المرتبطة به تمارس انتهاكات جسيمة وفظيعة لحقوق الإنسان بشكل يومي. إنهم يستهدفون بشكل منهجي رجالا ونساء وأطفالا بناء على إثنيتهم وولائهم الديني أو الطائفي".\n\nومضت بيلاي للقول إن مسيحيين وإيزيديين وتركمان من كانوا ضمن من استهدفهم تنظيم الدولة الإسلامية.\n\nمواضيع قد تهمك نهاية\n\nانتهاك منهجي\n\nوقالت مفوضة حقوق الإنسان إن حالات الاضطهاد والانتهاك المنهجي لحقوق ا

In [18]:
from lm_eval.models.huggingface import HFLM
if 'lm_obj' not in locals():
    kwargs = dict(
        pretrained=MODEL_PATH,
        trust_remote_code=True,
        parallelize=True,
        device_map="auto",
        tokenizer=TOKENIZER_PATH,
        batch_size=BATCH_SIZE,
    )

    if TUNED_MODEL_PATH:
        kwargs['peft'] = TUNED_MODEL_PATH

    lm_obj = HFLM(**kwargs)

2024-11-27:14:19:20,617 INFO     [huggingface.py:483] Using model type 'default'
2024-11-27:14:19:21,365 INFO     [huggingface.py:350] Model parallel was set to True, setting max memory per GPU to {0: 84523417600, 1: 84523417600} and device map to 'auto'


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [19]:
def evaluate_tasks(tasks,dataset_sub_path=TAWJEEH_DATASET_NAME):
    # MAKE SURE THE NOTEBOOK IS RUNNING FROM THE PROJECT ROOT!
    task_manager = TaskManager(include_path=f"eval_harness_extra_tasks/{dataset_sub_path}")
    results = simple_evaluate(  # call simple_evaluate
        model=lm_obj,
        tasks=tasks,
        num_fewshot=0,
        task_manager=task_manager,
    )
    return results

In [20]:
import json

def create_and_evaluate_single_prompt(prompt, save_results=True, force_re_evaluate=False):
    prompt_id = prompt['id']
    if TUNED_MODEL_PATH:
        results_dir = f'evaluation_results/{MODEL_NAME}-tuned/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    else:
        results_dir = f'evaluation_results/{MODEL_NAME}/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    prompt_results_file_path = f'{results_dir}/prompt_{prompt_id}.json'
    
    # Check if results exist and handle based on parameters
    if os.path.exists(prompt_results_file_path) and os.path.getsize(prompt_results_file_path) > 0:
        if not force_re_evaluate:
            print(f"Skipping prompt {prompt_id} - results already exist")
            with open(prompt_results_file_path, 'r') as f:
                prompt_results = json.load(f)
                print(make_table(prompt_results))
                return prompt_results
        else:
            print(f"Force re-evaluate enabled - reevaluating prompt {prompt_id}")
    
    # Create dataset and task files
    dataset = create_hf_dataset(prompt)
    
    # Save dataset
    dataset_dir = f'experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}'
    os.makedirs(dataset_dir, exist_ok=True)
    dataset['test'].to_parquet(f"{dataset_dir}/data.parquet")
    
    # Create YAML configuration
    yaml_text = f'''task: {TAWJEEH_DATASET_NAME}_prompt_{prompt_id}
dataset_path: experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}
output_type: generate_until
test_split: train
doc_to_text: text
doc_to_target: summary
metric_list:
  - metric: !function metrics.calculate_bleu
    aggregation: !function metrics.calculate_blue_agg
    higher_is_better: true
generation_kwargs:
    until:
    - <|eot_id|>
    - <|end_of_text|>
metadata:
  version: 1.0'''
    
    # Save YAML
    yaml_dir = f'eval_harness_extra_tasks/{TAWJEEH_DATASET_NAME}'
    os.makedirs(yaml_dir, exist_ok=True)
    with open(f'{yaml_dir}/prompt_{prompt_id}.yaml', 'w') as f:
        f.write(yaml_text)
    
    # Evaluate single prompt
    evaluation_task_name = f'{TAWJEEH_DATASET_NAME}_prompt_{prompt_id}'
    prompt_results = evaluate_tasks(tasks=[evaluation_task_name])
    
    print(make_table(prompt_results))
    
    # Save results if save_results is True
    if save_results:
        os.makedirs(results_dir, exist_ok=True)
        with open(prompt_results_file_path, 'w') as f:
            json.dump(prompt_results, f, ensure_ascii=False, indent=4, 
                     default=lambda o: '<not serializable>')
        print(f"Saved results for prompt {prompt_id}")
    else:
        print(f"Results not saved for prompt {prompt_id} (save_results=False)")
    
    print(f"Completed evaluation for prompt {prompt_id}")
    return prompt_results

In [21]:
def evaluate_all_prompts_sequentially(dataset_prompts, **kwargs):
    print(f"Starting sequential evaluation of {len(dataset_prompts)} prompts")
    all_results = {}
    
    for i, prompt in enumerate(dataset_prompts, 1):
        print('-' * 80)
        print(f"\nProcessing prompt {i}/{len(dataset_prompts)} (ID: {prompt['id']})")
        print("Template:", prompt['template'])
        print('-' * 80)
        
        prompt_results = create_and_evaluate_single_prompt(prompt, **kwargs)
        all_results[f"{TAWJEEH_DATASET_NAME}_prompt_{prompt['id']}"] = prompt_results
    
    return {'results': all_results}

# Evaluate

In [22]:
all_results = evaluate_all_prompts_sequentially(dataset_prompts=dataset_prompts)

Starting sequential evaluation of 5 prompts
--------------------------------------------------------------------------------

Processing prompt 1/5 (ID: 14871)
Template: You are an Arabic summarization expert! The summarization of the following Arabic passage: "{{text}}" is:
|||
{{target}}
--------------------------------------------------------------------------------
Skipping prompt 14871 - results already exist
|      Tasks       |Version|Filter|n-shot|    Metric    |   |Value |   |Stderr|
|------------------|------:|------|-----:|--------------|---|-----:|---|------|
|xlsum_prompt_14871|      1|none  |     0|calculate_bleu|↑  |5.8205|±  |   N/A|

--------------------------------------------------------------------------------

Processing prompt 2/5 (ID: 14858)
Template: Generate a summary for the following Arabic document: {{text}}
|||
{{target}}
--------------------------------------------------------------------------------
Skipping prompt 14858 - results already exist
|      Tas

In [ ]:
exit()

: 